# Transfer Learning, Embeddings & U-Net — Runnable Lab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jeevchiran/learnings-ai-ml/blob/main/notebook/transfer-learning/transfer-learning-lab.ipynb)

Companion notebook for the **Transfer Learning, Embeddings & U-Net** track
(`tl-m1` … `tl-m12`). It follows the same arc: take a pretrained backbone, use
it as a classifier, then as an embedding model, then rebuild it into a
segmentation network.

Everything runs on **CPU in a few minutes**. A GPU makes the two training
sections faster but nothing here requires one. No dataset downloads — the
segmentation data is generated in-notebook, and the classification demo uses
CIFAR-10 which torchvision fetches automatically.

Numbers the modules quote are re-derived here and checked with `assert`.

| Part | Modules | What runs |
|---|---|---|
| 1 | tl-m1 – tl-m3 | freezing, parameter budgets, BatchNorm, a real fine-tune |
| 2 | tl-m4 – tl-m5 | embeddings, cosine vs L2, kNN retrieval, triplet & InfoNCE |
| 3 | tl-m6 – tl-m9 | bottlenecks, four upsampling rules, transposed conv, skips |
| 4 | tl-m10 – tl-m12 | U-Net built, Dice/IoU verified, trained, threshold-swept |

## Setup

Colab already ships PyTorch, torchvision and scikit-learn, so the cell below
normally installs nothing. It only fetches a package if it is genuinely missing,
which keeps the notebook runnable outside Colab.

**Do not** `pip install torch` unconditionally on Colab — that can replace the
preinstalled CUDA build with a CPU-only wheel and silently lose your GPU.

In [ ]:
# Install only what is actually missing. On Colab this is a no-op and the
# existing CUDA build is left alone.
import importlib.util, subprocess, sys

for pkg, module in [('torch', 'torch'), ('torchvision', 'torchvision'),
                    ('scikit-learn', 'sklearn'), ('matplotlib', 'matplotlib')]:
    if importlib.util.find_spec(module) is None:
        print(f'installing {pkg} …')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=True)
    else:
        print(f'{pkg:<14} already present')

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, math
import matplotlib.pyplot as plt

torch.manual_seed(0); np.random.seed(0)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
plt.rcParams['figure.figsize'] = (11, 3.2)

print('torch      ', torch.__version__)
print('torchvision', __import__('torchvision').__version__)
print('device     ', DEVICE)
if DEVICE == 'cuda':
    print('gpu        ', torch.cuda.get_device_name(0))
else:
    print('note        running on CPU — everything works, the two training')
    print('            sections are just slower. Runtime → Change runtime type')
    print('            → T4 GPU to speed them up.')

---
# Part 1 — Transfer learning

## tl-m1 / tl-m2 · Freezing, and what it costs

The freeze decision is a **data budget**: every trainable parameter is one more
thing your labels have to pin down. Start by measuring it rather than guessing.

In [ ]:
import torchvision

def param_report(model, label=''):
    tr = sum(p.numel() for p in model.parameters() if p.requires_grad)
    to = sum(p.numel() for p in model.parameters())
    print(f'{label:<28} trainable {tr:>12,} / {to:>12,}  ({tr/to:6.2%})')
    return tr, to

m = torchvision.models.resnet18(weights=None)      # weights=None: no download needed here
param_report(m, 'resnet18, nothing frozen')

for p in m.parameters():
    p.requires_grad = False
m.fc = nn.Linear(m.fc.in_features, 10)             # new head is trainable by default
tr, to = param_report(m, 'frozen + new 10-way head')
assert tr == 512 * 10 + 10 == 5130                 # the tl-m3 figure
assert to == 11_181_642

for p in m.layer4.parameters():
    p.requires_grad = True
tr2, _ = param_report(m, '+ layer4 unfrozen')
print(f'\nunfreezing one block multiplied the trainable count by {tr2/tr:.0f}x')
assert tr2 == 8_398_858

**The BatchNorm trap.** `requires_grad = False` freezes *weights*. BatchNorm's
`running_mean` and `running_var` are **buffers**, updated in the forward pass,
so a "frozen" backbone still drifts. The next cell demonstrates it rather than
asserting it.

In [ ]:
def bn_stats(model):
    bn = [mod for mod in model.modules() if isinstance(mod, nn.BatchNorm2d)][0]
    return bn.running_mean.clone(), bn.running_var.clone()

net = torchvision.models.resnet18(weights=None)
for p in net.parameters():
    p.requires_grad = False

x = torch.randn(8, 3, 64, 64)

net.train()                                  # training mode: BN updates its buffers
m0, v0 = bn_stats(net); _ = net(x); m1, v1 = bn_stats(net)
drift_train = (m1 - m0).abs().max().item()

net.eval()                                   # eval mode: buffers are frozen
m2, v2 = bn_stats(net); _ = net(x); m3, v3 = bn_stats(net)
drift_eval = (m3 - m2).abs().max().item()

print(f'running_mean drift, model.train()  : {drift_train:.6f}   <- "frozen" model changed')
print(f'running_mean drift, model.eval()   : {drift_eval:.6f}   <- actually frozen')
assert drift_train > 0 and drift_eval == 0

def freeze_bn(module):
    """Call AFTER model.train() — .train() puts every BN back into training mode."""
    for mod in module.modules():
        if isinstance(mod, nn.BatchNorm2d):
            mod.eval()
            mod.weight.requires_grad = False
            mod.bias.requires_grad = False

## tl-m3 · A real fine-tune

CIFAR-10, downscaled to a few hundred images so it runs on CPU in a minute. The
point is not the accuracy — it is the **comparison**: a linear probe on frozen
features against the same architecture trained from scratch.

In [ ]:
from torchvision import transforms
from torch.utils.data import DataLoader, Subset

IMAGENET_MEAN, IMAGENET_STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
tf = transforms.Compose([
    transforms.Resize(64),                                  # small for CPU speed
    transforms.ToTensor(), transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

train_full = torchvision.datasets.CIFAR10('./data', train=True,  download=True, transform=tf)
test_full  = torchvision.datasets.CIFAR10('./data', train=False, download=True, transform=tf)

# a deliberately SMALL training set — the regime where transfer learning matters
train_ds = Subset(train_full, range(2000))
test_ds  = Subset(test_full,  range(1000))
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
test_loader  = DataLoader(test_ds,  batch_size=128)
print(f'{len(train_ds)} train / {len(test_ds)} test images')

In [ ]:
@torch.no_grad()
def extract(backbone, loader):
    backbone.eval()
    F_, Y_ = [], []
    for x, y in loader:
        F_.append(backbone(x.to(DEVICE)).cpu()); Y_.append(y)
    return torch.cat(F_).numpy(), torch.cat(Y_).numpy()

# tl-m4's key line: removing the classifier turns the net into an embedding model
backbone = torchvision.models.resnet18(weights='DEFAULT')
backbone.fc = nn.Identity()
backbone.to(DEVICE).eval()

Xtr, ytr = extract(backbone, train_loader)
Xte, yte = extract(backbone, test_loader)
print('feature shape:', Xtr.shape, ' <- 512-d embedding per image')
assert Xtr.shape[1] == 512

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier

# BASELINE 1 — majority class
dummy = DummyClassifier(strategy='most_frequent').fit(Xtr, ytr)
print(f'majority class            {dummy.score(Xte, yte):.4f}')

# BASELINE 2 — linear probe on FROZEN pretrained features
probe = LogisticRegression(max_iter=3000, n_jobs=-1).fit(Xtr, ytr)
probe_acc = probe.score(Xte, yte)
print(f'linear probe (frozen)     {probe_acc:.4f}')

# BASELINE 3 — same features from an UNTRAINED backbone (is pretraining doing the work?)
rand_backbone = torchvision.models.resnet18(weights=None)
rand_backbone.fc = nn.Identity(); rand_backbone.to(DEVICE).eval()
Xtr_r, _ = extract(rand_backbone, train_loader)
Xte_r, _ = extract(rand_backbone, test_loader)
rand_acc = LogisticRegression(max_iter=3000, n_jobs=-1).fit(Xtr_r, ytr).score(Xte_r, yte)
print(f'linear probe (random init) {rand_acc:.4f}')

print(f'\npretraining is worth {probe_acc - rand_acc:+.3f} accuracy here,')
print('on 2,000 images and with NO backbone training at all.')
assert probe_acc > rand_acc, 'pretrained features should beat random ones'

---
# Part 2 — Embeddings

## tl-m4 · Vector representations

The embedding is the vector; everything else is geometry on it.

In [ ]:
Xtr_t = torch.tensor(Xtr)
emb = F.normalize(Xtr_t, dim=1)               # L2-normalise → unit sphere
print('norms after normalisation:', emb.norm(dim=1)[:5].numpy().round(6))

# On unit vectors: ||a-b||^2 = 2 - 2*cos(a,b), so the two metrics rank identically.
a, b = emb[0], emb[1]
lhs = (a - b).pow(2).sum().item()
rhs = 2 - 2 * torch.dot(a, b).item()
print(f'||a-b||^2 = {lhs:.6f}   2 - 2cos = {rhs:.6f}')
assert abs(lhs - rhs) < 1e-5

sims = emb @ emb.T                            # the whole similarity matrix, one matmul
print('similarity matrix:', tuple(sims.shape))

In [ ]:
CLASSES = train_full.classes

def retrieve(q, k=5):
    s = sims[q].clone(); s[q] = -2                      # exclude the query itself
    idx = s.topk(k).indices
    return idx, s[idx]

# Does the neighbourhood agree with the label? That is retrieval quality
# without training a single parameter.
hits = 0
for q in range(len(emb)):
    idx, _ = retrieve(q, 5)
    hits += (ytr[idx.numpy()] == ytr[q]).mean()
print(f'mean precision@5 over {len(emb)} queries: {hits/len(emb):.4f}')
print(f'chance level would be {1/len(CLASSES):.4f}')

q = 0
idx, sc = retrieve(q, 5)
print(f'\nquery: {CLASSES[ytr[q]]}')
for i, s in zip(idx.numpy(), sc.numpy()):
    print(f'   {CLASSES[ytr[i]]:<12} cos={s:.4f}')

## tl-m5 · Metric learning

Triplet loss has three regimes and only two of them produce a gradient.

In [ ]:
def triplet(a, p, n, margin=0.2):
    return torch.clamp((a-p).norm(dim=-1) - (a-n).norm(dim=-1) + margin, min=0)

a = torch.tensor([0.0, 0.0])
cases = {
    'easy      ': (torch.tensor([0.4, 0.0]), torch.tensor([0.9, 0.0])),
    'semi-hard ': (torch.tensor([0.4, 0.0]), torch.tensor([0.5, 0.0])),
    'hard      ': (torch.tensor([0.8, 0.0]), torch.tensor([0.7, 0.0])),
}
for name, (p, n) in cases.items():
    L = triplet(a, p, n).item()
    print(f'{name} d(a,p)={ (a-p).norm():.2f}  d(a,n)={(a-n).norm():.2f}  loss={L:.3f}'
          f'   {"NO gradient" if L == 0 else "gradient flows"}')
assert triplet(a, *cases['easy      ']).item() == 0

# The sampling problem, measured: how many random triplets are already satisfied?
lab = torch.tensor(ytr)
g = torch.Generator().manual_seed(0)
zero = 0; N = 4000
for _ in range(N):
    i = torch.randint(len(emb), (1,), generator=g).item()
    pos = (lab == lab[i]).nonzero().flatten()
    neg = (lab != lab[i]).nonzero().flatten()
    p = pos[torch.randint(len(pos), (1,), generator=g)].item()
    n = neg[torch.randint(len(neg), (1,), generator=g)].item()
    if triplet(emb[i], emb[p], emb[n], 0.2).item() == 0:
        zero += 1
print(f'\n{zero}/{N} random triplets ({zero/N:.1%}) have ZERO loss on these embeddings.')
print('That is why triplet training needs mining, not sampling.')

In [ ]:
def info_nce(z1, z2, tau=0.07):
    """One positive (the diagonal) against every other item in the batch."""
    z1, z2 = F.normalize(z1, dim=1), F.normalize(z2, dim=1)
    logits = z1 @ z2.T / tau
    labels = torch.arange(len(z1), device=z1.device)
    return F.cross_entropy(logits, labels)

z = torch.randn(32, 64)
identical = info_nce(z, z.clone())
shuffled  = info_nce(z, z[torch.randperm(32)])
print(f'InfoNCE, perfectly matched views : {identical:.4f}')
print(f'InfoNCE, shuffled views          : {shuffled:.4f}')
assert identical < shuffled

for tau in (0.01, 0.07, 0.5, 5.0):
    print(f'  tau={tau:<5} loss={info_nce(z, z + 0.3*torch.randn_like(z), tau):.4f}')
print('\nSmall tau concentrates gradient on the hardest negative — mining, as a scalar.')

---
# Part 3 — Decoders

## tl-m6 · What the bottleneck costs

In [ ]:
size, ch = 128, 3
print(f'{"stage":<10}{"size":>10}{"ch":>6}{"values":>12}')
print(f'{"input":<10}{size:>10}{ch:>6}{size*size*ch:>12,}')
vals0 = size*size*ch
for d in range(4):
    ch = 32 * 2**d; size //= 2
    print(f'{"enc "+str(d+1):<10}{size:>10}{ch:>6}{size*size*ch:>12,}')
latent = 128
print(f'{"latent":<10}{1:>10}{latent:>6}{latent:>12,}')
print(f'\ncompression {vals0/latent:.0f}x — {100*latent/vals0:.2f}% of the values survive')

# The bound on boundary precision (tl-m6 worked example)
for inp, stages in [(256, 4), (512, 5), (384, 4)]:
    bott = inp // 2**stages
    print(f'{inp}px through {stages} stride-2 stages -> bottleneck {bott}px, '
          f'each cell covers {(inp//bott)**2} original pixels')
assert 512 // 2**5 == 16 and (512//16)**2 == 1024

## tl-m7 · The four fixed upsampling rules

None has a learnable parameter. Only one restores *position*.

In [ ]:
src = torch.tensor([[[[20., 60, 200, 180],
                      [40., 90, 220, 160],
                      [210., 190, 50, 30],
                      [180., 170, 70, 25]]]])

pooled, idx = F.max_pool2d(src, 2, return_indices=True)
print('pooled:\n', pooled[0,0].numpy())

nearest  = F.interpolate(pooled, scale_factor=2, mode='nearest')
bilinear = F.interpolate(pooled, scale_factor=2, mode='bilinear', align_corners=False)
unpooled = F.max_unpool2d(pooled, idx, 2)

for name, t in [('nearest', nearest), ('bilinear', bilinear), ('max-unpool', unpooled)]:
    err = (t - src).abs().mean().item()
    nz  = (t != 0).sum().item()
    print(f'{name:<12} mean|err| vs original {err:7.2f}   non-zero cells {nz}/16')

print('\nmax-unpool put each value back at its argmax position:')
print(unpooled[0,0].numpy())
assert (unpooled != 0).sum().item() == 4        # only the four maxima survive

In [ ]:
# align_corners: the half-pixel that shifts your masks.
x = torch.tensor([[[[0., 1.]]]])
f = F.interpolate(x, size=(1, 4), mode='bilinear', align_corners=False)
t = F.interpolate(x, size=(1, 4), mode='bilinear', align_corners=True)
print('align_corners=False:', f.flatten().numpy().round(4))
print('align_corners=True :', t.flatten().numpy().round(4))
print('\nDifferent sampling grids. Pick one and use it at BOTH training and export.')
assert not torch.allclose(f, t)

## tl-m8 · Transposed convolution

Scatter-add: each input cell stamps a scaled kernel into the output, and
overlaps sum. Where the stamp count varies, you get checkerboarding.

In [ ]:
def stamp_counts(N, k, s):
    """How many kernel stamps land on each output cell — ones in, ones kernel."""
    w = torch.ones(1, 1, k, k)
    out = F.conv_transpose2d(torch.ones(1, 1, N, N), w, stride=s)
    return out[0, 0]

print('interior stamp counts (border ramp excluded):')
for k, s in [(3, 2), (2, 2), (4, 2), (3, 1)]:
    c = stamp_counts(6, k, s)
    b = max(0, k - s)
    interior = c[b:c.shape[0]-b, b:c.shape[1]-b] if b else c
    uniq = sorted(set(interior.flatten().tolist()))
    print(f'  k={k} s={s}  out={tuple(c.shape)}  interior={uniq}  '
          f'{"CHECKERBOARDS" if len(uniq) > 1 else "clean"}')

# k divisible by s => every interior cell gets exactly (k/s)^2 stamps
c = stamp_counts(6, 4, 2); interior = c[2:-2, 2:-2]
assert set(interior.flatten().tolist()) == {4.0} == {(4/2)**2}

In [ ]:
# Output-size arithmetic, and what output_padding resolves.
def conv_out(n, k, s, p):  return (n + 2*p - k)//s + 1
def ct_out(n, k, s, p, op=0): return (n-1)*s - 2*p + k + op

print('a conv maps SEVERAL input sizes onto the same output:')
for n in (7, 8):
    print(f'  conv({n}, k=3, s=2, p=1) -> {conv_out(n,3,2,1)}')
print('so inverting the shape is ambiguous; output_padding picks which:')
for op in (0, 1):
    print(f'  convT(4, k=3, s=2, p=1, output_padding={op}) -> {ct_out(4,3,2,1,op)}')

# verify against the real op
for op in (0, 1):
    y = nn.ConvTranspose2d(1, 1, 3, stride=2, padding=1, output_padding=op)(torch.zeros(1,1,4,4))
    assert y.shape[-1] == ct_out(4,3,2,1,op)

up = nn.ConvTranspose2d(1024, 512, kernel_size=2, stride=2)      # U-Net's up-conv
print(f'\nU-Net up-conv: 28 -> {ct_out(28,2,2,0)},  '
      f'{sum(p.numel() for p in up.parameters()):,} parameters')
assert sum(p.numel() for p in up.parameters()) == 2_097_664

## tl-m9 · Skip connections

Two mechanisms, one name. Residual carries **gradient**; concatenative carries
**detail**. The next cell measures the first and sizes the second.

In [ ]:
D = 32
STD = 0.1        # linear gain ~ std*sqrt(D) = 0.57 < 1, so the plain path contracts

class Block(nn.Module):
    def __init__(self, d, residual):
        super().__init__()
        self.lin = nn.Linear(d, d, bias=False); self.residual = residual
        nn.init.normal_(self.lin.weight, std=STD)
    def forward(self, x):
        y = torch.tanh(self.lin(x))
        return x + y if self.residual else y

def grad_at_input(depth, residual):
    """Gradient magnitude arriving at the INPUT, for a fixed scalar loss.
    .sum() rather than .pow(2).mean() so the loss scale does not itself
    change with depth and confound the comparison."""
    net = nn.Sequential(*[Block(D, residual) for _ in range(depth)])
    x = torch.randn(4, D, requires_grad=True)
    net(x).sum().backward()
    return x.grad.abs().mean().item()

print(f'{"depth":>6}{"plain":>14}{"residual":>14}{"ratio":>14}')
for d in (2, 8, 20, 40):
    torch.manual_seed(0); p = grad_at_input(d, False)
    torch.manual_seed(0); r = grad_at_input(d, True)
    print(f'{d:>6}{p:>14.3e}{r:>14.3e}{r/p:>14.2e}')

print('\nPlain: each layer multiplies the gradient by roughly the same factor < 1,')
print('so it decays GEOMETRICALLY — by depth 40 it is ~1e-10 and those layers')
print('have effectively stopped learning.')
print('Residual: stays order 1 at every depth, because d/dx (x + f(x)) = 1 + f\'(x)')
print('always contains an unattenuated identity term. (It drifts mildly upward')
print('rather than staying pinned at 1 — the f\'(x) terms still accumulate.)')

torch.manual_seed(0); p40 = grad_at_input(40, False)
torch.manual_seed(0); r40 = grad_at_input(40, True)
assert p40 < 1e-6, 'the plain path should have vanished by depth 40'
assert r40 > 0.5,  'the residual path should still be order 1'

In [ ]:
# Concatenation costs exactly 2x the next conv's parameters — and buys independent weights.
up_ch, skip_ch, out_ch = 128, 128, 128
cat_conv = nn.Conv2d(up_ch + skip_ch, out_ch, 3, padding=1)
add_conv = nn.Conv2d(up_ch,           out_ch, 3, padding=1)
c, a = sum(p.numel() for p in cat_conv.parameters()), sum(p.numel() for p in add_conv.parameters())
print(f'concat: {c:,} parameters')
print(f'add   : {a:,} parameters   ({c/a:.2f}x cheaper)')
assert c == 295_040 and a == 147_584

x_up, x_skip = torch.randn(1, 128, 16, 16), torch.randn(1, 128, 16, 16)
print('\nconcat on dim=1 (channels):', torch.cat([x_skip, x_up], dim=1).shape)
print('concat on dim=0 (batch!)  :', torch.cat([x_skip, x_up], dim=0).shape, ' <- the classic bug')

---
# Part 4 — U-Net

## tl-m10 · Building it

Shape-first: run a random tensor through before touching data. Every U-Net bug
is a shape bug.

In [ ]:
class DoubleConv(nn.Module):
    """(conv3x3 -> BN -> ReLU) x2. padding=1 keeps the size, so no cropping."""
    def __init__(self, i, o):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(i, o, 3, padding=1, bias=False), nn.BatchNorm2d(o), nn.ReLU(inplace=True),
            nn.Conv2d(o, o, 3, padding=1, bias=False), nn.BatchNorm2d(o), nn.ReLU(inplace=True))
    def forward(self, x): return self.net(x)

class UNet(nn.Module):
    def __init__(self, in_ch=3, classes=1, base=32, depth=4):
        super().__init__()
        self.downs, self.ups = nn.ModuleList(), nn.ModuleList()
        self.pool = nn.MaxPool2d(2)
        ch = in_ch
        for d in range(depth):
            self.downs.append(DoubleConv(ch, base * 2**d)); ch = base * 2**d
        self.bottleneck = DoubleConv(ch, ch * 2)
        for d in reversed(range(depth)):
            o = base * 2**d
            self.ups.append(nn.ConvTranspose2d(o * 2, o, 2, stride=2))
            self.ups.append(DoubleConv(o * 2, o))      # upsampled o + skip o
        self.head = nn.Conv2d(base, classes, 1)

    def forward(self, x):
        skips = []
        for down in self.downs:
            x = down(x); skips.append(x); x = self.pool(x)   # save BEFORE pooling
        x = self.bottleneck(x)
        skips = skips[::-1]
        for i in range(0, len(self.ups), 2):
            x = self.ups[i](x)
            s = skips[i//2]
            if x.shape[-2:] != s.shape[-2:]:
                x = F.interpolate(x, size=s.shape[-2:], mode='bilinear', align_corners=False)
            x = self.ups[i+1](torch.cat([s, x], dim=1))      # concat on CHANNELS
        return self.head(x)                                  # raw logits

net = UNet(3, 1, base=32)
y = net(torch.randn(2, 3, 128, 128))
print('output shape:', tuple(y.shape), ' <- must match the input H,W')
print(f'{sum(p.numel() for p in net.parameters()):,} parameters')
assert y.shape == (2, 1, 128, 128)

In [ ]:
# The paper's configuration, counted exactly (tl-m10).
def conv_params(ci, co, k, bias=True): return k*k*ci*co + (co if bias else 0)

def unet_paper_params(base=64, depth=4, in_ch=1, classes=2):
    p, ch = 0, in_ch
    for d in range(depth):
        o = base * 2**d
        p += conv_params(ch, o, 3) + conv_params(o, o, 3); ch = o
    bott = base * 2**depth
    p += conv_params(ch, bott, 3) + conv_params(bott, bott, 3); ch = bott
    for d in reversed(range(depth)):
        o = base * 2**d
        p += conv_params(ch, o, 2)                       # 2x2 up-conv
        p += conv_params(2*o, o, 3) + conv_params(o, o, 3)
        ch = o
    return p + conv_params(ch, classes, 1)

print(f'original U-Net parameters: {unet_paper_params():,}')
assert unet_paper_params() == 31_030_658

# and the 572 -> 388 shrinkage from unpadded convs
size = 572
for _ in range(4):
    size -= 4; size //= 2
size -= 4                                                # bottleneck
for _ in range(4):
    size *= 2; size -= 4
print(f'572 in -> {size} out   (unpadded convs lose 4 px per level)')
assert size == 388

## tl-m11 · Dice and IoU

In [ ]:
def dice_coef(pred, gt, eps=0.0):
    inter = (pred * gt).sum()
    return ((2*inter + eps) / (pred.sum() + gt.sum() + eps)).item()

def iou_coef(pred, gt, eps=0.0):
    inter = (pred * gt).sum()
    return ((inter + eps) / (pred.sum() + gt.sum() - inter + eps)).item()

gt   = torch.tensor([[1., 1, 0], [1, 1, 0], [0, 0, 0]])
pred = torch.tensor([[1., 1, 0], [1, 0, 0], [0, 0, 0]])
d, j = dice_coef(pred, gt), iou_coef(pred, gt)
print(f'Dice {d:.4f}   IoU {j:.4f}   2J/(1+J) = {2*j/(1+j):.4f}')
assert abs(d - 2*j/(1+j)) < 1e-6

print(f'\n{"IoU":>6}{"Dice":>8}')
for j_ in (0.5, 0.6, 0.75, 0.8, 0.85, 0.9):
    print(f'{j_:>6.2f}{2*j_/(1+j_):>8.3f}')
print('\nDice always reads higher — never compare a Dice against someone else\'s IoU.')

In [ ]:
# Why BCE alone collapses under imbalance (tl-m11).
N = 64
yy, xx = torch.meshgrid(torch.arange(N), torch.arange(N), indexing='ij')

def make_gt(fg_frac):
    r = math.sqrt(fg_frac * N * N / math.pi)
    return (((xx - N/2)**2 + (yy - N/2)**2) <= r*r).float()

def soft_dice_loss(probs, gt, eps=1.0):
    inter = (probs * gt).sum()
    return 1 - ((2*inter + eps) / (probs.sum() + gt.sum() + eps)).item()

print(f'{"fg %":>6}{"BCE(empty)":>12}{"BCE(try)":>10}{"BCE picks":>12}{"Dice picks":>12}')
for fg in (0.02, 0.04, 0.10, 0.25, 0.45):
    g = make_gt(fg)
    empty   = torch.full_like(g, 0.01)
    attempt = torch.where(g > 0, torch.tensor(0.65), torch.tensor(0.35))
    b_e = F.binary_cross_entropy(empty, g).item()
    b_a = F.binary_cross_entropy(attempt, g).item()
    d_e, d_a = soft_dice_loss(empty, g), soft_dice_loss(attempt, g)
    print(f'{fg*100:>6.0f}{b_e:>12.4f}{b_a:>10.4f}'
          f'{("EMPTY" if b_e < b_a else "attempt"):>12}'
          f'{("EMPTY" if d_e < d_a else "attempt"):>12}')
print('\nBelow ~9% foreground BCE prefers predicting nothing. Dice never does,')
print('because an empty mask has zero intersection whatever the image size.')

g = make_gt(0.04)
empty = torch.full_like(g, 0.01)
attempt = torch.where(g > 0, torch.tensor(0.65), torch.tensor(0.35))
assert F.binary_cross_entropy(empty, g) < F.binary_cross_entropy(attempt, g)
assert soft_dice_loss(empty, g) > soft_dice_loss(attempt, g)

## tl-m12 · Training it

A synthetic segmentation dataset — blobs on a noisy background — generated in
memory. Small enough to train on CPU, real enough that the loss, the metric and
the threshold sweep all behave the way they do on real data.

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

def make_dataset(n, size=64, seed=0):
    g = torch.Generator().manual_seed(seed)
    X = torch.zeros(n, 3, size, size); Y = torch.zeros(n, 1, size, size)
    yy, xx = torch.meshgrid(torch.arange(size), torch.arange(size), indexing='ij')
    for i in range(n):
        img = 0.35 + 0.10 * torch.randn(3, size, size, generator=g)
        mask = torch.zeros(size, size)
        for _ in range(torch.randint(1, 4, (1,), generator=g).item()):
            cx, cy = torch.randint(14, size-14, (2,), generator=g).tolist()
            r = torch.randint(6, 13, (1,), generator=g).item()
            blob = (((xx-cx)**2 + (yy-cy)**2) <= r*r).float()
            mask = torch.clamp(mask + blob, 0, 1)
            img += 0.55 * blob.unsqueeze(0)                 # blobs are brighter
        X[i] = img.clamp(0, 1); Y[i, 0] = mask
    return TensorDataset(X, Y)

train_seg = make_dataset(240, seed=1)
val_seg   = make_dataset(60,  seed=2)
tl_loader = DataLoader(train_seg, batch_size=8, shuffle=True)
vl_loader = DataLoader(val_seg,   batch_size=8)

fg = torch.stack([y for _, y in val_seg]).mean().item()
print(f'{len(train_seg)} train / {len(val_seg)} val, foreground {fg:.1%}')

fig, ax = plt.subplots(2, 4, figsize=(11, 5.4))
for i in range(4):
    x, y = train_seg[i]
    ax[0, i].imshow(x.permute(1, 2, 0)); ax[0, i].axis('off')
    ax[1, i].imshow(y[0], cmap='gray');  ax[1, i].axis('off')
ax[0, 0].set_title('image', fontsize=9); ax[1, 0].set_title('mask', fontsize=9)
plt.tight_layout(); plt.show()

In [ ]:
def soft_dice_loss_t(logits, targets, eps=1.0):
    probs = torch.sigmoid(logits); dims = (1, 2, 3)
    inter = (probs*targets).sum(dims)
    return (1 - (2*inter + eps) / (probs.sum(dims) + targets.sum(dims) + eps)).mean()

bce_fn = nn.BCEWithLogitsLoss()
def criterion(logits, y): return 0.5*bce_fn(logits, y) + 0.5*soft_dice_loss_t(logits, y)

@torch.no_grad()
def dice_metric(logits, targets, thr=0.5, eps=1e-7):
    preds = (torch.sigmoid(logits) > thr).float(); dims = (1, 2, 3)
    inter = (preds*targets).sum(dims)
    return ((2*inter + eps) / (preds.sum(dims) + targets.sum(dims) + eps)).mean().item()

model = UNet(3, 1, base=16).to(DEVICE)          # base=16 keeps CPU training quick
opt = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-4)
EPOCHS = 8
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
print(f'{sum(p.numel() for p in model.parameters()):,} parameters')

history = []
for ep in range(EPOCHS):
    model.train()
    for x, y in tl_loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        loss = criterion(model(x), y)
        opt.zero_grad(); loss.backward(); opt.step()
    sched.step()
    model.eval()
    with torch.no_grad():
        ds = [dice_metric(model(x.to(DEVICE)), y.to(DEVICE)) for x, y in vl_loader]
    val = sum(ds)/len(ds); history.append(val)
    print(f'epoch {ep:>2}  loss {loss.item():.4f}  val Dice {val:.4f}')

print(f'\nbest val Dice {max(history):.4f}')
assert max(history) > 0.5, 'the U-Net should comfortably segment these blobs'

In [ ]:
# tl-m12's threshold sweep — free accuracy that almost everyone skips.
model.eval()
with torch.no_grad():
    L = torch.cat([model(x.to(DEVICE)).cpu() for x, _ in vl_loader])
    T = torch.cat([y for _, y in vl_loader])

sweep = {round(t, 2): dice_metric(L, T, thr=t) for t in np.arange(0.1, 0.91, 0.05)}
best_t = max(sweep, key=sweep.get)
for t, d in sweep.items():
    bar = '#' * int(d * 40)
    print(f'  {t:.2f}  {d:.4f}  {bar}{"  <-- best" if t == best_t else ""}')
print(f'\nbest threshold {best_t} (Dice {sweep[best_t]:.4f}) vs 0.5 (Dice {sweep[0.5]:.4f})')
print(f'moving the threshold is worth {sweep[best_t] - sweep[0.5]:+.4f} Dice, for zero training.')

In [ ]:
# Look at the predictions, not just the number.
with torch.no_grad():
    x, y = next(iter(vl_loader))
    pr = torch.sigmoid(model(x.to(DEVICE))).cpu()

fig, ax = plt.subplots(3, 4, figsize=(11, 7.6))
for i in range(4):
    ax[0, i].imshow(x[i].permute(1, 2, 0));            ax[0, i].axis('off')
    ax[1, i].imshow(y[i, 0], cmap='gray', vmin=0, vmax=1); ax[1, i].axis('off')
    ax[2, i].imshow((pr[i, 0] > best_t).float(), cmap='gray', vmin=0, vmax=1); ax[2, i].axis('off')
for a, t in zip(ax[:, 0], ['image', 'ground truth', f'predicted @{best_t}']):
    a.set_title(t, fontsize=9)
plt.tight_layout(); plt.show()

print('Per-image Dice for this batch:',
      [round(dice_metric(model(x[i:i+1].to(DEVICE)), y[i:i+1].to(DEVICE), best_t), 3) for i in range(4)])

---
## Where to go next

- **Pretrained encoder.** `segmentation_models_pytorch` swaps the contracting
  path for a pretrained ResNet in one argument — the single highest-value change
  on a small dataset, and it closes the loop back to Part 1.
- **Better augmentation.** `albumentations` applies the same geometric transform
  to image and mask together; elastic deformation is what made the original
  U-Net work on 30 images.
- **Multi-class.** Swap the 1-channel head for `C` channels, `BCEWithLogitsLoss`
  for `CrossEntropyLoss`, and reduce Dice per class then average (macro-Dice).
- **Boundary-aware losses.** Only once BCE+Dice has plateaued and the remaining
  error is edge placement rather than region.

Re-run with `base=32` and more epochs and every number above improves — but the
*shape* of the threshold sweep, and the reason Dice beats BCE, will not change.